<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">اگر آینده را باز بگذاریم چه می‌شود؟</h1><p style="text-align:right"><b>پرسش آزمایش:</b> آیا تغییر آینده می‌تواند خروجیِ گذشته را عوض کند؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html"><bdi dir="ltr">33-mask</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/34-causal-test.html"><bdi dir="ltr">34-causal-test</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این بار از <bdi dir="ltr">CausalSelfAttention</bdi> واقعی <bdi dir="ltr">Mini-GPT</bdi> استفاده می‌کنیم. وزن‌ها ثابت و آموزش‌ندیده‌اند، <bdi dir="ltr">Dropout</bdi> صفر است. آزمون دربارهٔ مسیر اطلاعات است، نه کیفیت زبان. پیش از اجرا حدس بزنید با تغییر دو موقعیت آخر، دو خروجی نخست در کدام حالت باید ثابت بمانند.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
config = ModelConfig(vocab_size=8, context_length=6, embedding_dim=8,
                     num_heads=2, num_layers=1, dropout=0.)
attention = CausalSelfAttention(config).eval()
x = torch.randn(1,4,8)
changed = x.clone()
changed[:,2:] += 5 * torch.randn_like(changed[:,2:])
results = {}
for causal in (False,True):
    trace = {}
    with torch.no_grad():
        original = attention(x, causal=causal, trace=trace)
        modified = attention(changed, causal=causal)
    difference = (original[:,:2]-modified[:,:2]).abs().max().item()
    print("causal:",causal,"prefix maximum change:",difference)
    inspect("actual allowed mask",trace["mask"])
    print(trace["mask"][0,0])
    results[causal] = trace
    if causal:
        torch.testing.assert_close(original[:,:2],modified[:,:2],rtol=0,atol=1e-7)
    else:
        assert difference > 1e-5


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(8,3))
for ax, causal in zip(axes,(False,True)):
    weights = results[causal]["weights"][0,0]
    ax.imshow(weights, vmin=0,vmax=1,cmap="Blues")
    ax.set(title=f"Causal = {causal}",xlabel="Key position",ylabel="Query position",
           xticks=range(4),yticks=range(4))
plt.tight_layout()
plt.show()
allowed = results[True]["mask"]
scores = results[True]["scaled_scores"]
wrong_weights = scores.masked_fill(~allowed, 0).softmax(-1)
print("Forbidden weight when scores are zeroed:", wrong_weights[0,0,0,1:].sum().item())
assert wrong_weights[0,0,0,1:].sum() > 0


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>انتظار و تمرین:</b> با <bdi dir="ltr">Mask</bdi>، خانه‌های آینده دقیقاً وزن صفر دارند. بدون آن، وقتی <bdi dir="ltr">Target</bdi> یک <bdi dir="ltr">Token</bdi> جلوتر از ورودی است، ورودیِ موقعیت بعد همان <bdi dir="ltr">Target</bdi> فعلی است؛ راهی برای نشت پاسخ باز می‌شود. پایین‌آمدن <bdi dir="ltr">Loss</bdi> در این حالت شاهد مدل علّی بهتر نیست. به‌جای صفرکردن امتیازها، چرا منفی بی‌نهایت می‌گذاریم؟ آزمایش را با <bdi dir="ltr">T</bdi>=6 بازنویسی کنید و مرز پیشوندِ ثابت را مشخص نگه دارید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/34-causal-test.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: نقشهٔ تجربی وابستگی موقعیت‌ها</h2>
<p style="text-align:right">به‌جای فقط دیدن <bdi dir="ltr">Mask</bdi>، با تغییر تک‌تک ورودی‌ها مسیر اثر را اندازه بگیرید. پیش‌نیاز: آزمون تغییر آینده و حالت <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">eval</code> در همین دفتر را بشناسید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">در ماتریسی که سطرش موقعیت خروجی و ستونش ورودیِ تغییرکرده است، اثرهای مجاز یک مدل علّی در کدام سمت قطر قرار می‌گیرند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
attention = CausalSelfAttention(ModelConfig(8,6,4,1,1,0.)).eval()
with torch.no_grad():
    attention.qkv.weight.zero_(); attention.qkv.bias.zero_()
    attention.qkv.weight[8:].copy_(torch.eye(4))
    attention.output.weight.copy_(torch.eye(4)); attention.output.bias.zero_()
x = torch.arange(16.).reshape(1,4,4)
print("Controlled fixture: uniform allowed weights; identity Value/output")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">dependency_matrix(attention,x,causal=True)</code> یک <bdi dir="ltr">Tensor</bdi> <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">bool</code> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(T,T)</code> برگرداند. برای هر ستون <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">j</code>، فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x[0,j,0]</code> را یک واحد زیاد کنید؛ سطر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">i</code> زمانی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">True</code> باشد که بیشینهٔ تغییر ویژگی‌های خروجی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">i</code> بیش از 1e-6 است. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x</code> را درجا تغییر ندهید؛ مدل همان شیء در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">eval</code> است.</p>
</div>

In [ ]:
def dependency_matrix(attention, x, causal=True):
    # TODO
    return None

In [ ]:
def test_exercise():
    before = x.clone()
    result = dependency_matrix(attention,x)
    if result is None: return False
    assert result.dtype == torch.bool
    assert torch.equal(result,torch.ones(4,4,dtype=torch.bool).tril())
    assert torch.equal(x,before)
    assert dependency_matrix(attention,x,causal=False).all()
    assert torch.equal(dependency_matrix(attention,x[:,:2]),torch.ones(2,2,dtype=torch.bool).tril())
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">causal</code> را خاموش کنید و اثر تغییر آخرین ورودی را بسنجید. این وزن‌های کنترل‌شده عمداً مسیرهای مجاز را قابل مشاهده کرده‌اند؛ در یک مدل دلخواه نبودِ اثر عددی، نبودِ مسیر را ثابت نمی‌کند.</p>
</div>

In [ ]:
changed = x.clone(); changed[0,-1,0] += 1
with torch.no_grad():
    for causal in (False,True):
        print(causal,(attention(changed,causal=causal)-attention(x,causal=causal)).abs().amax(-1))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">گزارش خراب سطر و ستون را برعکس تفسیر می‌کند و گذشته را نشت می‌نامد. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">has_future_influence(matrix)</code> با قرارداد سطر=خروجی، ستون=ورودیِ تغییرکرده، یک <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">bool</code> برگرداند.</p>
</div>

In [ ]:
observed = torch.ones(4,4,dtype=torch.bool).tril()
print('wrong test of allowed past:',observed.tril(-1).any().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def has_future_influence(matrix):
    # TODO
    return None

In [ ]:
def test_repair():
    result = has_future_influence(observed)
    if result is None: return False
    assert result is False
    assert has_future_influence(torch.ones(3,3,dtype=torch.bool)) is True
    assert has_future_influence(torch.eye(3,dtype=torch.bool)) is False
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">آزمایش روی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">CausalSelfAttention</code> واقعی انجام شد. نقشهٔ وابستگی، مکمل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">trace</code> وزن‌هاست؛ معنی هر دو محور باید در هر دو نمایش ثابت بماند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">اگر در یک مدل آموزش‌دیده یک خانهٔ مجاز اثر عددی صفر داشت، چرا نباید فوراً نتیجه بگیریم مسیر آن در معماری وجود ندارد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/34-causal-test.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-08_causal_mask.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>